In [34]:
from pyspark.sql import SparkSession

In [35]:
spark = SparkSession.builder.appName("Sales Summary validation").getOrCreate()


In [36]:
gold_df = spark.read.parquet("../../data/gold/sales_summary")

In [37]:
gold_df.printSchema()

root
 |-- order_date: date (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_items_sold: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- total_product_sales: double (nullable = true)
 |-- total_freight: double (nullable = true)
 |-- avg_order_value: double (nullable = true)



In [38]:
print("Gold rows:", gold_df.count())

Gold rows: 615


In [39]:
from pyspark.sql.functions import col, sum
gold_df.select(
    sum(col("order_date").isNull().cast("int")).alias("null_dates"),
    sum(col("total_revenue").isNull().cast("int")).alias("null_revenue")
).show()

+----------+------------+
|null_dates|null_revenue|
+----------+------------+
|         0|           0|
+----------+------------+



In [40]:
gold_df.filter(
    col("total_revenue") < 0
).show()

+----------+------------+----------------+-------------+-------------------+-------------+---------------+
|order_date|total_orders|total_items_sold|total_revenue|total_product_sales|total_freight|avg_order_value|
+----------+------------+----------------+-------------+-------------------+-------------+---------------+
+----------+------------+----------------+-------------+-------------------+-------------+---------------+



In [41]:
gold_df.filter(
    col("total_orders") <=0
).show()

+----------+------------+----------------+-------------+-------------------+-------------+---------------+
|order_date|total_orders|total_items_sold|total_revenue|total_product_sales|total_freight|avg_order_value|
+----------+------------+----------------+-------------+-------------------+-------------+---------------+
+----------+------------+----------------+-------------+-------------------+-------------+---------------+



In [42]:
gold_df.filter(
    col("total_items_sold") < col("total_orders")
).show()

+----------+------------+----------------+-------------+-------------------+-------------+---------------+
|order_date|total_orders|total_items_sold|total_revenue|total_product_sales|total_freight|avg_order_value|
+----------+------------+----------------+-------------+-------------------+-------------+---------------+
+----------+------------+----------------+-------------+-------------------+-------------+---------------+



In [43]:
orders = spark.read.parquet("../../data/silver/orders")
order_items = spark.read.parquet("../../data/silver/order_items")
payments = spark.read.parquet("../../data/silver/payments")

In [44]:
payments.groupBy("order_id").count().orderBy(col("count").desc()).show(10)

+--------------------+-----+
|            order_id|count|
+--------------------+-----+
|fa65dad1b0e818e3c...|   29|
|ccf804e764ed5650c...|   26|
|285c2e15bebd4ac83...|   22|
|895ab968e7bb0d565...|   21|
|fedcd9f7ccdc8cba3...|   19|
|ee9ca989fc93ba09a...|   19|
|21577126c19bf11a0...|   15|
|4bfcba9e084f46c8e...|   15|
|3c58bffb70dcf45f1...|   14|
|4689b1816de42507a...|   14|
+--------------------+-----+
only showing top 10 rows


In [45]:
order_items.groupBy("order_id") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(10)

+--------------------+-----+
|            order_id|count|
+--------------------+-----+
|8272b63d03f5f79c5...|   21|
|1b15974a0141d54e3...|   20|
|ab14fdcfbe524636d...|   20|
|428a2f660dc84138d...|   15|
|9ef13efd6949e4573...|   15|
|9bdc4d4c71aa1de46...|   14|
|73c8ab38f07dc9438...|   14|
|37ee401157a3a0b28...|   13|
|af822dacd6f5cff73...|   12|
|2c2a19b5703863c90...|   12|
+--------------------+-----+
only showing top 10 rows
